# [6.1] SAE Variants - Solutions

Reference validation notebook for the section-local SAE variant implementation. This executes the visible tests against `solutions.py`, then checks the CPU notebook contract and the committed CUDA report highlights.

Expected CUDA highlights: pinned Pythia-70M hidden-state extraction runs on CUDA, a tiny TopK-16 SAE improves held-out reconstruction over a zero baseline, the permuted-decoder negative control is worse, held-out feature AUC clears the threshold, decoder steering beats an orthogonal random control, and peak VRAM stays below the configured budget.


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter6_sparse_feature_methods"
section = "part1_sae_variants"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_sae_variants.tests as tests
from part1_sae_variants import solutions


In [ ]:
tests.test_encoder_variants_match_reference_and_sparsity_rules(
    solutions.relu_l1_encode,
    solutions.topk_encode,
    solutions.gated_encode,
    solutions.jumprelu_encode,
)
tests.test_decode_and_metrics_match_identity_contract(
    solutions.decode_features,
    solutions.sae_variant_metrics,
    solutions.feature_density,
    solutions.l0,
    solutions.dead_feature_fraction,
)
tests.test_toy_superposition_batch_has_planted_sparse_structure(
    solutions.make_toy_superposition_batch,
    solutions.density_is_nondegenerate,
)
tests.test_dictionary_recovery_detects_duplicates_and_missing_features(
    solutions.dictionary_recovery_report,
)
tests.test_best_feature_auc_handles_predictive_and_antipredictive_features(
    solutions.roc_auc_binary,
    solutions.best_feature_auc,
)
tests.test_decoder_steering_changes_last_position_and_reports_control(
    solutions.apply_decoder_steering,
    solutions.steering_comparison_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)


In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
smoke


In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"], "Pythia hidden-state TopK SAE preflight should pass."
assert gpu["model_id"] == "EleutherAI/pythia-70m-deduped", "The CUDA path should use the pinned public Pythia checkpoint."
assert gpu["revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c", "The Pythia checkpoint revision should remain pinned."
assert gpu["reconstruction_improvement_vs_zero"] >= 0.15, "Held-out reconstruction should improve over the zero baseline."
assert gpu["random_decoder_control_passed"], "Permuted decoder negative control should reconstruct worse."
assert gpu["density_nondegenerate"], "Sparse feature density should be nondegenerate."
assert gpu["best_feature_auc"] >= 0.95, "Held-out feature AUC should clear the release threshold."
assert gpu["passes_decoder_steering_control"], "Decoder-vector steering should beat the orthogonal random control."
assert gpu["peak_vram_gb"] <= 1.0, "Tiny SAE preflight should stay under the locked VRAM budget."
{
    "heldout_reconstruction_mse": gpu["heldout_reconstruction_mse"],
    "reconstruction_improvement_vs_zero": gpu["reconstruction_improvement_vs_zero"],
    "random_decoder_mse_ratio": gpu["random_decoder_mse_ratio"],
    "best_feature_auc": gpu["best_feature_auc"],
    "safe_logit_delta": gpu["safe_logit_delta"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
